## ЗАДАЧА 1: Установка библиотек

In [1]:
   import sys, torch, transformers
   print(sys.executable)
   print(torch.__version__, transformers.__version__)

/Users/mak/Desktop/WORKPLACE/ML_валютная удаленка/Проекты/transformers/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


/Users/mak/Desktop/WORKPLACE/ML_валютная удаленка/Проекты/transformers/.venv/bin/python
2.15.0 5.21.0


In [2]:
print('hello')

hello


In [3]:
from transformers import AutoTokenizer
import torch

## ЗАДАЧА 2: Загрузка токенизатора

In [4]:
model_name = "distilbert-base-uncased"

In [5]:
#грузим токенизатор
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [6]:
#смотрим параметры
print(tokenizer.vocab_size)
print(tokenizer.model_max_length)

30522
512


## ЗАДАЧА 3: Токенизация текста
Пытаемся как-то поработать с тектом

In [7]:
text = "This movie was absolutely amazing!"

In [8]:
tokens = tokenizer(text)
print(tokens)

{'input_ids': [101, 2023, 3185, 2001, 7078, 6429, 999, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1]}


In [9]:
input_ids = tokens['input_ids']
print(f'Количество токенов: {len(input_ids)}')

Количество токенов: 8


In [10]:
decoded = tokenizer.decode(input_ids)
print(f'Декодировано: {decoded}')

Декодировано: [CLS] this movie was absolutely amazing! [SEP]


## ЗАДАЧА 4: Работа с батчами

In [11]:
def tokenize_texts(texts, max_length=128):
    return tokenizer(
        texts,
        padding=True, #вот из-за этого у нас у разных текстов будет одинаковое кол-во токенов!
        truncation=True,
        max_length=max_length,
        return_tensors="pt"
    )

In [12]:
texts = [
    "This movie was too much yey yey yey!",
    "Terrible movie, waste of time.",
    "I love you!"
]

In [13]:
tokens = tokenize_texts(texts)
print(f'Shape: {tokens["input_ids"].shape}')

Shape: torch.Size([3, 14])


In [14]:
#Маска говорит модели: «на позиции с 0 не смотри».
print(f'Attention mask:\n{tokens["attention_mask"]}')

Attention mask:
tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0]])


In [15]:
#как выглядит каждый токен внутри каждого текста
for t in texts:
    print(tokenizer.tokenize(t))

['this', 'movie', 'was', 'too', 'much', 'ye', '##y', 'ye', '##y', 'ye', '##y', '!']
['terrible', 'movie', ',', 'waste', 'of', 'time', '.']
['i', 'love', 'you', '!']


In [16]:
# до маски как было 
raw = tokenizer(texts)          # без padding=True
for ids in raw["input_ids"]:
    print(len(ids), ids)

14 [101, 2023, 3185, 2001, 2205, 2172, 6300, 2100, 6300, 2100, 6300, 2100, 999, 102]
9 [101, 6659, 3185, 1010, 5949, 1997, 2051, 1012, 102]
6 [101, 1045, 2293, 2017, 999, 102]


## ЗАДАЧА 5: Специальные токены

In [17]:
print(f'CLS token: {tokenizer.cls_token} (ID: {tokenizer.cls_token_id})')
print(f'SEP token: {tokenizer.sep_token} (ID: {tokenizer.sep_token_id})')
print(f'PAD token: {tokenizer.pad_token} (ID: {tokenizer.pad_token_id})')

CLS token: [CLS] (ID: 101)
SEP token: [SEP] (ID: 102)
PAD token: [PAD] (ID: 0)


In [18]:
single = tokenizer(text, return_tensors="pt")
print(f'Input IDs: {single["input_ids"]}')
print(f'Decoded: {tokenizer.decode(single["input_ids"][0])}')

Input IDs: tensor([[ 101, 2023, 3185, 2001, 7078, 6429,  999,  102]])
Decoded: [CLS] this movie was absolutely amazing! [SEP]


## ЗАДАЧА 6: Функция explain_tokenization

In [19]:
def explain_tokenization(text, tokenizer):
    tokens = tokenizer.tokenize(text)
    ids = tokenizer.convert_tokens_to_ids(tokens)

    print(f'Исходный текст: {text}')
    print(f'Токены: {tokens}')
    print(f'IDs: {ids}')
    print(f'Количество: {len(tokens)}')

In [20]:
explain_tokenization("Transformers are amazing!", tokenizer)

Исходный текст: Transformers are amazing!
Токены: ['transformers', 'are', 'amazing', '!']
IDs: [19081, 2024, 6429, 999]
Количество: 4


In [21]:
explain_tokenization("Tokenization is unbelievably fun", tokenizer)
explain_tokenization("Магнит", tokenizer)

Исходный текст: Tokenization is unbelievably fun
Токены: ['token', '##ization', 'is', 'un', '##bel', '##ie', '##va', '##bly', 'fun']
IDs: [19204, 3989, 2003, 4895, 8671, 2666, 3567, 6321, 4569]
Количество: 9
Исходный текст: Магнит
Токены: ['м', '##а', '##г', '##н', '##и', '##т']
IDs: [1191, 10260, 29741, 18947, 10325, 22919]
Количество: 6
